In [ ]:
# Import necesarios para construir el dataset
import pandas as pd
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
import numpy as np
import re
import random
import json
from datasets import Dataset, Features, Value, ClassLabel, Sequence

In [20]:
# Configuración del modelo y parámetros
MODEL_NAME = "gpt2-small"
TOKEN_NAME_SIZE = 1
DATASET_SIZE = len(names)
TEMPLATE_TYPE = "subject_with_name"
PROMPT_TYPE_SIZE = 50

# Cargar el modelo
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True
)

she_token = model.to_tokens(f" she", prepend_bos=False)[0].tolist()[0]
he_token = model.to_tokens(f" he", prepend_bos=False)[0].tolist()[0]

print(f"she_token: {she_token}, he_token: {he_token}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
she_token: 673, he_token: 339


In [ ]:
# Configuración de rutas
NAMES_FILEPATH = "../datasets/names.csv"

# Cargar datos de nombres
df = pd.read_csv(NAMES_FILEPATH)

# Filtrar nombres unisex (nombres con alta probabilidad tanto para hombre como mujer)
names = df[ (df["F_weighted_norm"] > 0.2) & (df["M_weighted_norm"] > 0.2) ]["name"].tolist()

In [21]:
# Crear un dataframe con los nombres unisex
unisex_df = pd.DataFrame({
    'name': names,
})

# Obtener información de género para cada nombre
name_gender_info = []
for name in names:
    gender_info = df[df['name'] == name][['M_weighted_norm', 'F_weighted_norm']].iloc[0]
    name_gender_info.append(gender_info)

unisex_df['M_weighted_norm'] = [info['M_weighted_norm'] for info in name_gender_info]
unisex_df['F_weighted_norm'] = [info['F_weighted_norm'] for info in name_gender_info]

print(f"Dataset unisex con {len(unisex_df)} nombres")

Dataset unisex con 27 nombres


In [22]:
def fill_prompt(prompt: str, fillers: dict) -> str:
    """
    Fills a prompt template by replacing placeholders with actual values.
    """
    elems = re.findall(r"\[(.*?)\]", prompt)
    for elem in elems:
        if elem in fillers:
            prompt = re.sub(rf"\[{elem}\]", random.choice(fillers[elem]), prompt)
    return prompt

def get_prompts(name: str, prompt: str) -> dict:
    """
    Genera prompts original y ablacionado para un nombre dado.
    """
    org_prompt = re.sub(r"\[name\]", name, prompt)
    ablated_prompt = re.sub(r"\[name\]", "someone", prompt)
    return {
        "org_prompt": org_prompt,
        "ablated_prompt": ablated_prompt
    }

def get_name_token_info(model: HookedTransformer, name: str, prompt: str) -> dict:
    """
    Returns token ids, token strings, and token positions for the name.
    """
    # Tokenize prompt
    prompt_tokens = model.to_tokens(prompt, prepend_bos=True)[0].tolist()

    # Tokenize the name
    name_tokens = model.to_tokens(f" {name}", prepend_bos=False)[0].tolist()
    name_token_strs = model.to_str_tokens(f" {name}", prepend_bos=False)
    
    # Find positions of name tokens in the prompt
    positions = [model.get_token_position(single_token=int(token), input=prompt, prepend_bos=True) for token in name_tokens]

    return {
        "subject": {
            "token_idxs": name_tokens,
            "tokens": name_token_strs,
            "pos": positions
        },
        "end": {
            "pos": len(prompt_tokens) - 1
        }
    }

print("Funciones de procesamiento definidas")

Funciones de procesamiento definidas


In [24]:
# Cargar templates
with open("../src/json/templates_he_she.json", "r", encoding="utf-8") as f:
    jdata = json.load(f)

templates = jdata["templates"][TEMPLATE_TYPE]["prompt_templates"]
complements = jdata["complements"]

print(f"Templates cargados: {len(templates)} templates disponibles")

Templates cargados: 3 templates disponibles


In [25]:
# Construir el dataset
dataset_list = []
dataset_id = 1

for prompt_type in range(PROMPT_TYPE_SIZE):
    # Seleccionar un template aleatorio y llenarlo
    template = random.choice(templates)
    filled_prompt_template = fill_prompt(template, complements)
    
    for idx, name in enumerate(unisex_df['name'].tolist()):
        # Obtener el género esperado
        m_norm = unisex_df.iloc[idx]['M_weighted_norm']
        f_norm = unisex_df.iloc[idx]['F_weighted_norm']
        
        # Asignar token esperado basado en género (para unisex, puede ser cualquiera)
        # En este caso, usaremos una distribución aleatoria
        expected_token_id = he_token if m_norm > f_norm else she_token
        
        # Obtener prompts
        prompts_dict = get_prompts(name, filled_prompt_template)
        
        # Obtener información de tokens
        try:
            token_info = get_name_token_info(model, name, prompts_dict["org_prompt"])
            
            # Construir el item del dataset
            item = {
                "id": dataset_id,
                "prompt_type": prompt_type,
                "prompts": prompts_dict,
                "subject": token_info["subject"],
                "end": token_info["end"],
                "expected_token_id": expected_token_id
            }
            
            dataset_list.append(item)
            dataset_id += 1
        except Exception as e:
            print(f"Error procesando nombre '{name}' en prompt_type {prompt_type}: {e}")

print(f"Dataset construido con {len(dataset_list)} instancias")

Dataset construido con 1350 instancias


In [26]:
# Ver ejemplo del dataset
if len(dataset_list) > 0:
    print("Ejemplo de item del dataset:")
    print(dataset_list[0])

Ejemplo de item del dataset:
{'id': 1, 'prompt_type': 0, 'prompts': {'org_prompt': 'Earlier today, Alexis (to the group) declared what', 'ablated_prompt': 'Earlier today, someone (to the group) declared what'}, 'subject': {'token_idxs': [31078], 'tokens': [' Alexis'], 'pos': [4]}, 'end': {'pos': 11}, 'expected_token_id': 673}


In [27]:
# Convertir a formato de Dataset de Hugging Face (opcional)
features = Features({
    "id": Value("int32"),
    "prompt_type": ClassLabel(names=[f"type_{i}" for i in range(PROMPT_TYPE_SIZE)]),
    "prompts": {
        "org_prompt": Value("string"),
        "ablated_prompt": Value("string")
    },
    "subject": {
        "token_idxs": Sequence(Value("int32")),
        "tokens": Sequence(Value("string")),
        "pos": Sequence(Value("int32"))
    },
    "end": {
        "pos": Value("int32")
    },
    "expected_token_id": Value("int32")
})

# Preparar datos para el Dataset
dataset_dict = {
    "id": [item["id"] for item in dataset_list],
    "prompt_type": [item["prompt_type"] for item in dataset_list],
    "prompts": [item["prompts"] for item in dataset_list],
    "subject": [item["subject"] for item in dataset_list],
    "end": [item["end"] for item in dataset_list],
    "expected_token_id": [item["expected_token_id"] for item in dataset_list]
}

unisex_dataset = Dataset.from_dict(dataset_dict, features=features)
print(f"Dataset de Hugging Face creado con {len(unisex_dataset)} ejemplos")

Dataset de Hugging Face creado con 1350 ejemplos


In [28]:
# Guardar el dataset
unisex_dataset.save_to_disk("../datasets/unisex_names_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 1350/1350 [00:00<00:00, 28599.26 examples/s]
